# XARVIS: train my voice

This page turns the recordings you made in XARVIS (**☰ → Record my voice → SEND RECORDINGS → Google Drive**) into a voice file that XARVIS speaks with, on your phone, offline.

**Before you start:** at the top, tap **Runtime → Change runtime type → T4 GPU → Save**.

Then press ▶ on each step, **one at a time, top to bottom**, and wait for the green ✓ before the next one.

- Step 1 asks to connect Google Drive: allow it (that's where your recordings are, and where the voice is saved).
- Step 4 is the long one (about 1 to 2 hours). Keep this page open. If it stops, just run Steps 1–4 again: it carries on where it left off.
- Do **Hindi** first. When it's finished, change LANGUAGE to **en** in Step 1 and run Steps 1–5 again for English.

Your recordings and your voice stay in your own Google Drive. Keep them private.

In [ ]:
#@title Step 1: settings and your recordings
LANGUAGE = "hi"  #@param ["hi", "en"]
#@markdown More rounds = closer to your voice, but longer. 600 is about 1–2 hours on the free T4 GPU.
EXTRA_EPOCHS = 600  #@param {type:"integer"}

import glob, os, subprocess, zipfile
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/XARVIS voice"
os.makedirs(OUT, exist_ok=True)

zips = sorted(glob.glob("/content/drive/MyDrive/**/xarvis-my-voice*.zip", recursive=True), key=os.path.getmtime)
assert zips, "I can't find xarvis-my-voice.zip in your Google Drive. In XARVIS: ☰ → Record my voice → SEND RECORDINGS → Drive."
print("Recordings:", zips[-1])
with zipfile.ZipFile(zips[-1]) as z:
    z.extractall("/content/data")
DATA = f"/content/data/{LANGUAGE}"
assert os.path.exists(f"{DATA}/metadata.csv"), f"The zip has no {LANGUAGE} recordings. Record that tab first, or pick the other LANGUAGE."
n = sum(1 for _ in open(f"{DATA}/metadata.csv", encoding="utf-8"))
print(f"{n} {'Hindi' if LANGUAGE == 'hi' else 'English'} sentences ✓")
if n < 60:
    print("That's few; the voice will sound rough. Around 120+ is much better.")
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
assert "GPU" in gpu, "No GPU. Runtime → Change runtime type → T4 GPU → Save, then run Step 1 again."
print(gpu)

In [ ]:
#@title Step 2: install the voice trainer (about 5–10 minutes)
%cd /content
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
![ -d piper1-gpl ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip -q install scikit-build "cmake>=3.18,<4" "ninja>=1,<2" huggingface_hub iso639-lang
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py -q build_ext --inplace
import importlib, sys
sys.path.insert(0, "/content/piper1-gpl/src")
print("Trainer installed ✓")

In [ ]:
#@title Step 3: get a starting voice to learn from
# Training starts from an existing Piper voice and moves it towards yours: much faster and
# better than starting from nothing. Hindi starts from a Hindi voice when one is available.
import pickle, types, torch
from huggingface_hub import HfApi, hf_hub_download

class _Anything:
    def __init__(self, *a, **k): pass
    def __setstate__(self, state): pass
class _Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            return _Anything
def epoch_of(path):
    """How many rounds a checkpoint has had (without needing the code that saved it)."""
    loose = types.SimpleNamespace(Unpickler=_Unpickler, load=pickle.load, __name__="pickle")
    return int(torch.load(path, map_location="cpu", weights_only=False, pickle_module=loose).get("epoch", 0))

files = HfApi().list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset")
ckpts = [f for f in files if f.endswith(".ckpt") and "/medium/" in f]
prefer = ["hi/hi_IN/"] if LANGUAGE == "hi" else ["en/en_GB/", "en/en_US/lessac/"]
prefer += ["en/en_US/lessac/", "en/en_US/"]
base = next(c for p in prefer for c in ckpts if c.startswith(p))
print("Starting from:", base)
BASE_CKPT = hf_hub_download("rhasspy/piper-checkpoints", base, repo_type="dataset")

RUN = f"{OUT}/{LANGUAGE}"
os.makedirs(RUN, exist_ok=True)
last = sorted(glob.glob(f"{RUN}/lightning_logs/*/checkpoints/last.ckpt"), key=os.path.getmtime)
START = last[-1] if last else BASE_CKPT
target_file = f"{RUN}/target_epochs.txt"
if os.path.exists(target_file):
    TARGET = int(open(target_file).read())
else:
    TARGET = epoch_of(BASE_CKPT) + EXTRA_EPOCHS
    open(target_file, "w").write(str(TARGET))
done = epoch_of(START)
print(f"Carrying on from where it stopped ({TARGET - done} rounds left) ✓" if last else f"Ready: {EXTRA_EPOCHS} rounds of training ✓")

In [ ]:
#@title Step 4: train (the long one: keep this page open)
%cd /content/piper1-gpl
ESPEAK = "hi" if LANGUAGE == "hi" else "en-us"
!python3 -m piper.train fit \
  --data.voice_name "rex" \
  --data.csv_path "{DATA}/metadata.csv" \
  --data.audio_dir "{DATA}/wavs" \
  --model.sample_rate 22050 \
  --data.espeak_voice "{ESPEAK}" \
  --data.cache_dir "/content/cache/{LANGUAGE}" \
  --data.config_path "{RUN}/config.json" \
  --data.batch_size 16 \
  --trainer.max_epochs {TARGET} \
  --trainer.default_root_dir "{RUN}" \
  --trainer.accelerator gpu --trainer.devices 1 \
  --ckpt_path "{START}"
print("Training finished ✓")

In [ ]:
#@title Step 5: make the voice file for XARVIS
# Turns the trained voice into what XARVIS's engine (sherpa-onnx) reads, then zips it into
# your Drive as "XARVIS voice/xarvis-voice-<language>.zip".
import json, onnx, shutil, tarfile, urllib.request
%cd /content/piper1-gpl
last = sorted(glob.glob(f"{RUN}/lightning_logs/*/checkpoints/last.ckpt"), key=os.path.getmtime)[-1]
pack = f"/content/pack/{LANGUAGE}"
shutil.rmtree(pack, ignore_errors=True)
os.makedirs(pack)
!python3 -m piper.train.export_onnx --checkpoint "{last}" --output-file "{pack}/model.onnx"

config = json.load(open(f"{RUN}/config.json", encoding="utf-8"))
with open(f"{pack}/tokens.txt", "w", encoding="utf-8") as f:
    for s, i in config["phoneme_id_map"].items():
        if s == "\n":
            continue
        f.write(f"{s} {i[0] if isinstance(i, list) else i}\n")
voice = config.get("espeak", {}).get("voice", "hi" if LANGUAGE == "hi" else "en-us")
meta = {
    "model_type": "vits", "comment": "piper", "language": "Hindi" if LANGUAGE == "hi" else "English",
    "voice": voice, "version": 1, "has_espeak": 1, "has_g2pw": 0,
    "n_speakers": config.get("num_speakers", 1), "sample_rate": config["audio"]["sample_rate"],
}
m = onnx.load(f"{pack}/model.onnx")
while len(m.metadata_props):
    m.metadata_props.pop()
for k, v in meta.items():
    p = m.metadata_props.add(); p.key = k; p.value = str(v)
onnx.save(m, f"{pack}/model.onnx")

# The pronunciation data the phone's engine uses (sherpa-onnx's own copy of espeak-ng-data).
urllib.request.urlretrieve("https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models/espeak-ng-data.tar.bz2", "/content/espeak.tar.bz2")
with tarfile.open("/content/espeak.tar.bz2") as t:
    t.extractall(pack)

open(f"{pack}/lang.txt", "w").write(LANGUAGE)
zip_path = shutil.make_archive(f"{OUT}/xarvis-voice-{LANGUAGE}", "zip", pack)
print(f"Done ✓  Your voice: Google Drive → XARVIS voice → xarvis-voice-{LANGUAGE}.zip ({os.path.getsize(zip_path) // 1_000_000} MB)")
print("In XARVIS: ☰ → Load my voice → pick that file.")